# 🛠️ Function Calling Basics

**Connect LLMs to external tools and APIs**

---

## 📋 Overview

**What you'll learn:**
- What is function calling?
- Defining functions for LLMs
- Executing function calls
- Error handling
- Real-world examples

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
import os
import json
from typing import Dict, List, Any

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 What is Function Calling?

### Before Function Calling:
```
User: "What's the weather in Paris?"
LLM:  "I don't have access to real-time weather data."
      ❌ Limited to training data
```

### With Function Calling:
```
User: "What's the weather in Paris?"
LLM:  → Calls get_weather(location="Paris")
API:  → Returns {temp: 18, condition: "Sunny"}
LLM:  "It's 18°C and sunny in Paris!"
      ✅ Real-time data access
```

### Key Concepts:

**1. Tool Definition**
```python
# You define what functions are available
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get weather for a location",
        "parameters": {...}
    }
}]
```

**2. LLM Decides**
```python
# LLM chooses when to call functions
# Extracts parameters from user query
```

**3. You Execute**
```python
# You run the actual function
# Return results to LLM
```

**4. LLM Responds**
```python
# LLM uses results to answer user
```

## 🎯 Simple Example: Weather API

In [ ]:
# Step 1: Define the function
def get_weather(location: str, unit: str = "celsius") -> Dict:
    """Get weather for a location (mock implementation)."""
    
    # In production, call real weather API
    weather_data = {
        "paris": {"temp": 18, "condition": "Sunny"},
        "london": {"temp": 12, "condition": "Rainy"},
        "tokyo": {"temp": 22, "condition": "Cloudy"},
    }
    
    location_lower = location.lower()
    
    if location_lower in weather_data:
        return weather_data[location_lower]
    else:
        return {"error": f"Weather data not available for {location}"}

# Step 2: Define tool schema for LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a specific location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city name, e.g., Paris, London"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    }
]

# Step 3: Make request with tools
user_query = "What's the weather like in Paris?"

messages = [{"role": "user", "content": user_query}]

response = client.chat.completions.create(
    model="gpt-4",
    messages=messages,
    tools=tools,
    tool_choice="auto"  # Let LLM decide when to use tools
)

response_message = response.choices[0].message

print("🎯 LLM Response:\n")
print(f"Message: {response_message.content}")
print(f"Tool calls: {response_message.tool_calls}")

# Step 4: Check if LLM wants to call a function
if response_message.tool_calls:
    tool_call = response_message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    
    print(f"\n🔧 Function Call:")
    print(f"  Name: {function_name}")
    print(f"  Arguments: {function_args}")
    
    # Step 5: Execute the function
    if function_name == "get_weather":
        function_result = get_weather(**function_args)
        print(f"  Result: {function_result}")
        
        # Step 6: Send result back to LLM
        messages.append(response_message)  # Assistant's request
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(function_result)
        })
        
        # Step 7: Get final response
        final_response = client.chat.completions.create(
            model="gpt-4",
            messages=messages
        )
        
        print(f"\n💬 Final Response:")
        print(final_response.choices[0].message.content)

## 🔄 Function Calling Loop

In [ ]:
# Available functions
available_functions = {
    "get_weather": get_weather,
}

def run_conversation(user_query: str, tools: List[Dict], max_iterations: int = 5) -> str:
    """Run a conversation with function calling."""
    
    messages = [{"role": "user", "content": user_query}]
    
    for iteration in range(max_iterations):
        print(f"\n🔄 Iteration {iteration + 1}")
        
        # Call LLM
        response = client.chat.completions.create(
            model="gpt-4",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        response_message = response.choices[0].message
        
        # Check if done (no tool calls)
        if not response_message.tool_calls:
            print("✅ No more tool calls - returning response")
            return response_message.content
        
        # Add assistant message to conversation
        messages.append(response_message)
        
        # Execute all tool calls
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"  🔧 Calling: {function_name}({function_args})")
            
            # Execute function
            if function_name in available_functions:
                function_response = available_functions[function_name](**function_args)
                print(f"     Result: {function_response}")
            else:
                function_response = {"error": f"Function {function_name} not found"}
            
            # Add tool response to conversation
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(function_response)
            })
    
    return "Max iterations reached"

# Test it
print("🔄 Running Conversation Loop\n")
print("="*60)

query = "What's the weather in Paris and London?"
print(f"User: {query}\n")

final_answer = run_conversation(query, tools)

print(f"\n💬 Final Answer:\n{final_answer}")

## 🛠️ Multiple Tools Example

In [ ]:
# Define multiple tools
def get_weather(location: str) -> Dict:
    """Get weather data."""
    return {"location": location, "temp": 20, "condition": "Sunny"}

def search_flights(origin: str, destination: str, date: str) -> Dict:
    """Search for flights."""
    return {
        "flights": [
            {"airline": "AirFrance", "price": 250, "time": "10:00"},
            {"airline": "EasyJet", "price": 120, "time": "14:30"},
        ]
    }

def book_hotel(location: str, check_in: str, check_out: str) -> Dict:
    """Book a hotel."""
    return {
        "hotel": "Grand Hotel",
        "confirmation": "ABC123",
        "price": 150
    }

# Tool definitions
multi_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather information",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name"}
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_flights",
            "description": "Search for available flights",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string"},
                    "destination": {"type": "string"},
                    "date": {"type": "string", "description": "Date in YYYY-MM-DD format"}
                },
                "required": ["origin", "destination", "date"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_hotel",
            "description": "Book a hotel room",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "check_in": {"type": "string"},
                    "check_out": {"type": "string"}
                },
                "required": ["location", "check_in", "check_out"]
            }
        }
    }
]

# Update available functions
multi_functions = {
    "get_weather": get_weather,
    "search_flights": search_flights,
    "book_hotel": book_hotel,
}

print("🛠️  Multiple Tools Example\n")
print("Available tools:")
for tool in multi_tools:
    print(f"  • {tool['function']['name']}: {tool['function']['description']}")

print("\n💬 Example Query:")
print("  'I want to fly from Paris to London on 2024-06-15 and need a hotel'")
print("\n💡 LLM will automatically choose which tools to call!")

## ✅ Summary

### Function Calling Flow:

```python
1. User query
   ↓
2. LLM receives query + available tools
   ↓
3. LLM decides to call function(s)
   ↓
4. You execute the function(s)
   ↓
5. Send results back to LLM
   ↓
6. LLM generates final response
```

### Tool Definition Structure:

```python
{
    "type": "function",
    "function": {
        "name": "function_name",
        "description": "What the function does",  # Be specific!
        "parameters": {
            "type": "object",
            "properties": {
                "param1": {
                    "type": "string",
                    "description": "What this param is"
                }
            },
            "required": ["param1"]
        }
    }
}
```

### Best Practices:

**1. Clear Descriptions**
```python
# ❌ Bad
"description": "Get weather"

# ✅ Good
"description": "Get current weather information for a specific city including temperature and conditions"
```

**2. Explicit Parameters**
```python
# ❌ Bad
"location": {"type": "string"}

# ✅ Good
"location": {
    "type": "string",
    "description": "City name, e.g., 'Paris' or 'London'",
    "examples": ["Paris", "New York"]
}
```

**3. Error Handling**
```python
def safe_function_call(function_name, args):
    try:
        return available_functions[function_name](**args)
    except Exception as e:
        return {"error": str(e)}
```

**4. Parallel Tool Calls**
```python
# LLM can call multiple tools at once
for tool_call in response_message.tool_calls:
    # Execute each one
    result = execute(tool_call)
```

### Common Use Cases:

✅ **Perfect for:**
- External APIs (weather, flights, stock prices)
- Database queries
- Calculations
- File operations
- System commands

❌ **Not ideal for:**
- Simple text transformations (use prompts)
- Data already in context
- High-latency operations (async better)

### Next: `07_agents_tools/02_tool_chains.ipynb`